# Lab 3. Hyperparameter optimization

Tune the best model of lab 2 with cross-validation and compare Grid Search with Random Search.

- **Task:** Titanic survival from [lab 2](../lab2/): the same `train.csv`, the same seven features, preprocessing and train/test split (`random_state=42`).
- **Model:** SVM with the RBF kernel, the best model of lab 2 by CV. Its hyperparameters have a clear geometric meaning.
- **Metric:** F1 for `Survived = 1`.
- The decision threshold chosen in lab 2 is not part of this lab: every model here uses the default threshold 0, so the comparison is about the hyperparameters only.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import loguniform
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    RepeatedStratifiedKFold,
    cross_validate,
    train_test_split,
    validation_curve,
)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

pd.set_option("display.notebook_repr_html", False)
pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 80)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 72

SEED = 42
TARGET = "Survived"
NUMERIC = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
CATEGORICAL = ["Sex", "Embarked"]
FEATURES = NUMERIC + CATEGORICAL

## 1. Setup and baseline

The **baseline** of this lab is SVM with default parameters (`C=1`, `gamma="scale"`, `class_weight=None`). Its CV F1 and test F1 are fixed before any tuning.

Its CV F1 here (0.730) differs from lab 2 (0.736) only because the CV scheme differs: 15 folds instead of 5 (section 3). The test F1 (0.793) is the same.

In [ ]:
data = pd.read_csv("../lab2/data/train.csv")
train, test = train_test_split(data, test_size=0.2, stratify=data[TARGET], random_state=SEED)
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

preprocessor = ColumnTransformer(
    [
        (
            "num",
            make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()),
            NUMERIC,
        ),
        (
            "cat",
            make_pipeline(
                SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")
            ),
            CATEGORICAL,
        ),
    ]
)
baseline = Pipeline([("prep", preprocessor), ("model", SVC(random_state=SEED))])
CV = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)


def short(params: dict) -> str:
    values = {k.removeprefix("model__"): v for k, v in sorted(params.items())}
    return ", ".join(
        f"{k}={v:.3g}" if isinstance(v, float) else f"{k}={v}" for k, v in values.items()
    )


baseline_folds = cross_validate(baseline, X_train, y_train, cv=CV, scoring="f1")["test_score"]
baseline_test = f1_score(y_test, clone(baseline).fit(X_train, y_train).predict(X_test))
print(f"train: {len(train)}, test: {len(test)}")
print(f"baseline (default SVM): CV F1 {baseline_folds.mean():.3f} +- {baseline_folds.std():.3f}")
print(f"baseline: test F1 {baseline_test:.3f}")

## 2. Hyperparameters

| Hyperparameter | Meaning | Range | Expected effect |
| --- | --- | --- | --- |
| `C` | the price of a training error compared with a wide margin | 0.01–1000, log scale | small → a wide, soft margin and a simple boundary, up to underfitting; large → the boundary bends around single passengers, overfitting |
| `gamma` | the width of the RBF kernel: how far the influence of one training passenger reaches | 0.001–10, log scale | small → a smooth, almost linear boundary; large → every passenger affects only a small neighborhood, the boundary turns into islands around training points, overfitting |
| `class_weight` | the weight of errors on each class | `None`, `"balanced"` | `"balanced"` makes an error on a survivor (the minority, 38%) cost about 1.6 times more than on a non-survivor → more recall, less precision |

`C` and `gamma` change the model by orders of magnitude, so both are searched on a log scale. They also work together: a large `C` with a small `gamma` and a small `C` with a larger `gamma` can give similar boundaries. The kernel itself is fixed (RBF): it was the one used in lab 2, and a linear kernel is close to RBF with a very small `gamma` and a large `C`. Both searches use exactly these ranges.

## 3. Cross-validation

**Scheme:** `RepeatedStratifiedKFold(n_splits=5, n_repeats=3)`, i.e. 15 estimates for every parameter set, computed on the **training part only**. The test set does not take part in the search.

- **Why 5 folds:** each model is trained on 570 passengers (80% of train, close to the final size) and validated on 142 (about 55 survivors). 10 repeated folds would double the number of fits (30 for each of 198 combinations), while the repeats already reduce the variance of the estimate.
- **Why 3 repeats:** the data is small, and the result depends on which passengers land in which fold. Three different splits average this out, and on 712 rows it is cheap.
- All models are evaluated on **the same 15 folds**, so they can be compared fold by fold (section 7).

**Answers**

- **Why cross-validation is needed:** to estimate quality on data the model did not see, using every training object for validation once in every repeat and averaging several estimates instead of trusting one.
- **Why one train/test split is not enough:** with 712 passengers, F1 on one validation part varies by ±0.04 depending on the split (the baseline std in section 1). That is larger than the difference between good parameter sets, so on one split we would simply pick the luckiest one.
- **What happens if we tune on test many times:** we fit the parameters to the 179 test passengers, the test score becomes optimistic, and there is no honest estimate of quality on new data left.

## 4. Grid Search

In [ ]:
grid_space = {
    "model__C": np.logspace(-2, 3, 11),
    "model__gamma": np.logspace(-3, 1, 9),
    "model__class_weight": [None, "balanced"],
}
grid = GridSearchCV(baseline, grid_space, cv=CV, scoring="f1", n_jobs=-1, return_train_score=True)
start = time.perf_counter()
grid.fit(X_train, y_train)
grid_time = time.perf_counter() - start

n_grid = len(grid.cv_results_["params"])
print(f"combinations: {n_grid}, fits: {n_grid * CV.get_n_splits()}")
print(f"time: {grid_time:.1f} s")
print(f"best CV F1: {grid.best_score_:.3f}")
print(f"best params: {short(grid.best_params_)}")

## 5. Random Search

In [ ]:
random_space = {
    "model__C": loguniform(1e-2, 1e3),
    "model__gamma": loguniform(1e-3, 1e1),
    "model__class_weight": [None, "balanced"],
}
search = RandomizedSearchCV(
    baseline,
    random_space,
    n_iter=n_grid,
    cv=CV,
    scoring="f1",
    n_jobs=-1,
    random_state=SEED,
    return_train_score=True,
)
start = time.perf_counter()
search.fit(X_train, y_train)
random_time = time.perf_counter() - start


def fit_time(searcher: GridSearchCV | RandomizedSearchCV) -> float:
    return searcher.cv_results_["mean_fit_time"].sum() * CV.get_n_splits()


pd.DataFrame(
    {
        "combinations": [n_grid, len(search.cv_results_["params"])],
        "time, s": [grid_time, random_time],
        "total fit time, s": [fit_time(grid), fit_time(search)],
        "best CV F1": [grid.best_score_, search.best_score_],
        "best params": [short(grid.best_params_), short(search.best_params_)],
    },
    index=["Grid Search", "Random Search"],
)

**Findings**

- **Grid Search** tried all 11 × 9 × 2 = **198 combinations** (2970 fits). Best CV F1 **0.745** with `C=1, gamma=0.316, class_weight="balanced"`.
- **Random Search** got the same budget: 198 random combinations from the same ranges (`C` and `gamma` log-uniform). Best CV F1 **0.742** with `C≈0.95, gamma≈0.25, "balanced"`, practically the same point.
- **Quality:** equal (difference 0.003, far below the fold spread of about 0.03–0.04).
- **Time:** of the same order. By the wall clock Random Search was faster, but the total time of the fits themselves (third column) is comparable, as it should be with the same number of fits. The wall-clock difference comes mostly from the first search also starting the parallel worker processes, and it varies from run to run.
- **Flat optimum:** very different settings reach almost the same CV, e.g. `C=1, gamma=0.316` (0.745) and `C=100, gamma=0.01` (0.741) (section 8). `C` and `gamma` compensate each other, so there is a whole band of good values rather than one point.

**When Random Search is better than a full grid:**

- **Many hyperparameters, few of them important.** A grid wastes most combinations repeating the same values of the important ones, while every random combination tries a new value of each parameter.
- **Continuous parameters** such as `C` and `gamma`: a grid checks only the listed points, a random search any value in the range. Here it found `C≈0.95, gamma≈0.25`, values the grid does not contain.
- **Limited time:** the number of iterations is set in advance and the search can be stopped at any moment, while a grid grows multiplicatively with every new parameter.

## 6. Influence of hyperparameters

In [ ]:
curves = {"C": np.logspace(-2, 3, 11), "gamma": np.logspace(-3, 1, 9)}
curve_tables = {}
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (param, values) in zip(axes, curves.items(), strict=True):
    train_scores, cv_scores = validation_curve(
        baseline,
        X_train,
        y_train,
        param_name=f"model__{param}",
        param_range=values,
        cv=CV,
        scoring="f1",
        n_jobs=-1,
    )
    for scores, label, color in [(train_scores, "train", "#d6604d"), (cv_scores, "CV", "#2166ac")]:
        mean, std = scores.mean(axis=1), scores.std(axis=1)
        ax.plot(values, mean, "o-", color=color, label=label)
        ax.fill_between(values, mean - std, mean + std, color=color, alpha=0.15)
    ax.set_xscale("log")
    ax.set_xlabel(param)
    ax.set_ylabel("F1")
    ax.set_title(f"{param} (others default)")
    ax.legend()
    curve_tables[param] = pd.DataFrame(
        {"train": train_scores.mean(axis=1), "CV": cv_scores.mean(axis=1)},
        index=[f"{v:.3g}" for v in values],
    ).T
fig.tight_layout()
plt.show()
for param, table in curve_tables.items():
    print(f"F1 by {param}")
    display(table.round(3))

In [ ]:
grid_table = pd.DataFrame(grid.cv_results_["params"]).rename(
    columns=lambda c: c.removeprefix("model__")
)
grid_table["class_weight"] = grid_table["class_weight"].fillna("None")
grid_table["CV F1"] = grid.cv_results_["mean_test_score"]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, weight in zip(axes, ["None", "balanced"], strict=True):
    heat = grid_table[grid_table["class_weight"] == weight].pivot_table(
        index="gamma", columns="C", values="CV F1"
    )
    sns.heatmap(
        heat.rename(index="{:.3g}".format, columns="{:.3g}".format),
        annot=True,
        fmt=".2f",
        cmap="RdBu",
        vmin=0.6,
        vmax=0.76,
        cbar=ax is axes[1],
        annot_kws={"size": 8},
        ax=ax,
    )
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=0)
    ax.set_title(f"CV F1, class_weight={weight}")
    ax.set_xlabel("C")
    ax.set_ylabel("gamma")
fig.tight_layout()
plt.show()

weights = {}
for weight in [None, "balanced"]:
    scores = cross_validate(
        clone(baseline).set_params(model__class_weight=weight),
        X_train,
        y_train,
        cv=CV,
        scoring=["precision", "recall", "f1"],
    )
    weights[f"class_weight={weight}"] = {
        metric: scores[f"test_{metric}"].mean() for metric in ["precision", "recall", "f1"]
    }
pd.DataFrame(weights).T

Each curve changes one hyperparameter while the others stay at their defaults (`C=1`, `gamma="scale"`, `class_weight=None`). The bands are ± one standard deviation over the 15 folds. The heatmaps show the mean CV F1 of every grid combination.

| Hyperparameter | Underfitting | Reasonable complexity | Overfitting |
| --- | --- | --- | --- |
| `C` | ≤ 0.03: F1 0–0.2 on both train and CV, the margin is so soft that the model predicts almost everyone as dead | 0.3–10: CV ≈ 0.72–0.73 | ≥ 100: train grows to 0.85–0.88 while CV falls to 0.68 at `C=1000` |
| `gamma` | 0.001: F1 ≈ 0.21 on both, the boundary is too smooth | 0.01–1: CV ≈ 0.71–0.73 | ≥ 3: train 0.88–0.90, CV drops to 0.65 and 0.54 |

**How to read the curves:**

- **Underfitting:** train and CV are both low. The model is too simple even for the training data.
- **Reasonable complexity:** CV is at its maximum, and the gap between train and CV is moderate.
- **Overfitting:** train keeps growing while CV goes down. The model memorizes individual passengers, which does not transfer to new ones.

**The heatmaps** show the interaction: the good values form a diagonal band from "large `C`, small `gamma`" to "small `C`, larger `gamma`". The left columns (tiny `C`) underfit, the lower right corner (large `C` and large `gamma`) overfits.

**`class_weight`** at the default `C` and `gamma` does not change F1 (0.730 vs 0.731) but trades precision for recall (precision 0.81 → 0.73, recall 0.67 → 0.73). It moves the boundary towards the non-survivors, much like the lower decision threshold in lab 2.

## 7. Final models

In [ ]:
def fold_scores(searcher: GridSearchCV | RandomizedSearchCV) -> np.ndarray:
    results, best = searcher.cv_results_, searcher.best_index_
    return np.array([results[f"split{i}_test_score"][best] for i in range(CV.get_n_splits())])


folds = {
    "Baseline (default)": baseline_folds,
    "Grid Search": fold_scores(grid),
    "Random Search": fold_scores(search),
}
test_predictions = {
    "Baseline (default)": clone(baseline).fit(X_train, y_train).predict(X_test),
    "Grid Search": grid.predict(X_test),
    "Random Search": search.predict(X_test),
}
counts = {name: confusion_matrix(y_test, pred).ravel() for name, pred in test_predictions.items()}
final = pd.DataFrame(
    {
        "CV F1": {name: scores.mean() for name, scores in folds.items()},
        "CV std": {name: scores.std() for name, scores in folds.items()},
        "CV min": {name: scores.min() for name, scores in folds.items()},
        "test F1": {name: f1_score(y_test, pred) for name, pred in test_predictions.items()},
        "TP": {name: c[3] for name, c in counts.items()},
        "FP": {name: c[1] for name, c in counts.items()},
        "FN": {name: c[2] for name, c in counts.items()},
    }
)
final

In [ ]:
paired = pd.DataFrame.from_dict(
    {
        name: {
            "mean gain vs baseline": (folds[name] - baseline_folds).mean(),
            "std of gain": (folds[name] - baseline_folds).std(),
            f"folds improved (of {len(baseline_folds)})": (folds[name] > baseline_folds).sum(),
        }
        for name in ["Grid Search", "Random Search"]
    },
    orient="index",
)
paired.round(3)

**Findings**

- Both tuned models are refitted on the whole training part after the search (`refit=True` in `GridSearchCV` and `RandomizedSearchCV`), and these are the models evaluated on the test.
- **Fold-by-fold comparison.** The ± std intervals overlap, but the models are evaluated on the same folds, so a fold-by-fold comparison is the right one (second table): the Grid Search model is better than the baseline on **12 of 15 folds**, by **+0.015** on average. The gain is small, and it is optimistic: the 15 folds reuse the same 712 passengers, and the winner of 198 configurations was chosen by this very CV.
- The tuned model has a slightly lower std (0.034 vs 0.040) and a higher worst fold (0.673 vs 0.627), but on 15 dependent folds this is within noise.
- **On the test F1 barely changes:** 0.793 for the baseline, 0.797 for Grid Search, 0.788 for Random Search. The answers themselves change noticeably: with Grid Search FP grow from 4 to 11 and FN fall from 21 to 16, so at least 12 passengers get a different answer. `class_weight="balanced"` trades precision for recall on the test as well, and in F1 the two changes almost cancel out.
- **Why the test does not repeat the CV gain:**
  1. *The test is small:* 179 passengers with 69 survivors; one passenger changes F1 by about 0.01.
  2. *Winner's curse:* the best of 198 CV scores is partly lucky; the configuration that won also happened to fit these folds well.
  3. *Few features:* the flat optimum suggests that with seven features the default SVM is already close to what they can tell about survival.
- In lab 2 the SVM with threshold −0.2 got 0.803 on the test. `class_weight="balanced"` does almost the same as a lower threshold: it moves the boundary towards the non-survivors. The difference between 0.803 and 0.797 is within the noise of the test.

## 8. Stability

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
sns.stripplot(
    data=pd.DataFrame(folds).melt(var_name="model", value_name="F1"),
    x="F1",
    y="model",
    size=7,
    alpha=0.7,
    ax=ax,
)
ax.set_title("F1 on each of the 15 CV folds")
plt.show()

top = pd.DataFrame(grid.cv_results_).sort_values("mean_test_score", ascending=False).head(6)
top["params"] = top["params"].map(short)
top[["params", "mean_test_score", "std_test_score", "mean_train_score"]].set_index("params")

**Findings**

- **The split matters more than the parameters.** The same model gets very different F1 on different folds (baseline from 0.63 to 0.79, the Grid Search model from 0.67 to 0.80), while the means differ by only 0.015.
- **Same mean, different stability.** The six best grid configurations have practically equal mean F1 (0.738–0.745), but their std ranges from 0.033 to 0.039. They also differ in the train–CV gap: the winner has train 0.809 vs CV 0.745, while `C=100, gamma=0.01` has train 0.764 vs CV 0.741. The second one is simpler (a smoother boundary) for almost the same CV.
- Two models with equal mean can differ in how much they depend on the particular data: one scores evenly, the other high on some folds and noticeably lower on others. On new data the second one is less predictable.
- **The selection rule** was fixed in advance: the highest mean CV score (the default of both searches). With differences this small, a reasonable alternative is to prefer the simplest or most stable configuration among those close to the best.
- In lab 2 the SVM had a CV std of 0.008, here 0.040: there were only 5 folds, and they happened to be lucky; 15 folds show the spread better.

## Conclusions

| Model | Parameters | CV F1 | CV std | Test F1 |
| --- | --- | --- | --- | --- |
| Baseline | `C=1`, `gamma="scale"`, `class_weight=None` | 0.730 | 0.040 | 0.793 |
| Grid Search | `C=1`, `gamma=0.316`, `class_weight="balanced"` | **0.745** | 0.034 | **0.797** |
| Random Search | `C≈0.95`, `gamma≈0.25`, `class_weight="balanced"` | 0.742 | **0.031** | 0.788 |

- Hyperparameters were chosen only by repeated 5-fold CV on train; the test was used once for each final model.
- Grid Search and Random Search found the same region with the same budget. Random Search reached values the grid does not contain; it is preferable when there are many parameters or continuous ranges.
- The validation curves show the usual picture: underfitting with a tiny `C` or `gamma`, overfitting with a large `C` or `gamma`, and a wide plateau in between. `C` and `gamma` compensate each other, and `class_weight` trades precision for recall.
- Tuning gave a small CV gain (+0.015, better on 12 of 15 folds). On the 179-passenger test the difference in F1 is within noise, although the answers changed: more survivors found and more FP.